<a href="https://colab.research.google.com/github/sr606/LLM/blob/main/Mermaid6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
class Graph:
    def __init__(self):
        self.sources = set()
        self.lookups = set()
        self.processing = set()
        self.outputs = set()
        self.edges = []  # (src, tgt, label)

    def add_source(self, name):
        self.sources.add(name)

    def add_lookup(self, name):
        self.lookups.add(name)

    def add_processing(self, name):
        self.processing.add(name)

    def add_output(self, name):
        self.outputs.add(name)

    def add_edge(self, src, tgt, label=None):
        self.edges.append((src, tgt, label))




from graph_model import Graph


def build_graph(lineage_response):
    graph = Graph()

    for stage in lineage_response.stages:

        stage_id = stage.stage_id

        # Processing node
        graph.add_processing(stage_id)

        # -------------------
        # Source Tables
        # -------------------
        for src in stage.source_tables:
            graph.add_source(src)
            graph.add_edge(src, stage_id)

        # -------------------
        # Lookup Tables
        # -------------------
        for lookup in stage.lookup_tables:
            graph.add_lookup(lookup)
            graph.add_edge(lookup, stage_id, label="Lookup")

        # -------------------
        # Transformations
        # Only meaningful ones
        # -------------------
        for tf in stage.transformations:
            label = tf.summary_label

            for src_table in tf.source_tables:
                graph.add_edge(
                    src_table,
                    stage_id,
                    label=label
                )

        # -------------------
        # Outputs
        # -------------------
        for out in stage.output_tables:
            graph.add_output(out)
            graph.add_edge(stage_id, out)

        # -------------------
        # Conditional Outputs (Exception paths)
        # -------------------
        for cond in stage.output_conditions:
            graph.add_output(cond.label)
            graph.add_edge(stage_id, cond.label, label="Condition")

    return graph

In [ ]:
#graph_model_replace


class Graph:
    def __init__(self):
        self.stages = {}          # stage_id -> stage object
        self.datasets = set()     # dataset names
        self.stage_edges = []     # (dataset → stage) and (stage → dataset)
        self.column_edges = []    # (source_column, target_column, expression)

    def add_stage(self, stage_id):
        self.stages[stage_id] = {}

    def add_dataset(self, dataset_name):
        self.datasets.add(dataset_name)

    def add_stage_edge(self, source, target):
        self.stage_edges.append((source, target))

    def add_column_edge(self, src_col, tgt_col, expression):
        self.column_edges.append((src_col, tgt_col, expression))



#only build_graph() function
    def build_graph(lineage_response):
    graph = Graph()

    for stage in lineage_response.stages:
        graph.add_stage(stage.stage_id)

        # Inputs → Stage
        for inp in stage.inputs:
            dataset = inp["dataset_name"]
            graph.add_dataset(dataset)
            graph.add_stage_edge(dataset, stage.stage_id)

        # Stage → Outputs
        for out in stage.outputs:
            dataset = out["dataset_name"]
            graph.add_dataset(dataset)
            graph.add_stage_edge(stage.stage_id, dataset)

        # Column lineage
        for col in stage.column_lineage:
            for src in col.source_columns:
                graph.add_column_edge(
                    src,
                    f"{stage.stage_id}.{col.target_column}",
                    col.expression
                )

    return graph



#renderer new
from graphviz import Digraph


def render_pipeline_view(graph, output_path):
    dot = Digraph(format="pdf")
    dot.attr(rankdir="LR")

    # Dataset nodes
    for dataset in graph.datasets:
        dot.node(dataset, shape="box", style="filled", fillcolor="#E3F2FD")

    # Stage nodes
    for stage in graph.stages:
        dot.node(stage, shape="ellipse", style="filled", fillcolor="#FFF3E0")

    # Edges
    for src, tgt in graph.stage_edges:
        dot.edge(src, tgt)

    dot.render(output_path, cleanup=True)



def render_stage_column_view(graph, stage_id, output_path):
    dot = Digraph(format="pdf")
    dot.attr(rankdir="LR")

    dot.node(stage_id, shape="ellipse", style="filled", fillcolor="#FFE0B2")

    for src, tgt, expr in graph.column_edges:
        if tgt.startswith(stage_id + "."):
            dot.node(src, shape="box", fillcolor="#E8F5E9", style="filled")
            dot.node(tgt, shape="box", fillcolor="#FFF9C4", style="filled")

            label = expr[:40] + "..." if len(expr) > 40 else expr
            dot.edge(src, tgt, label=label)

    dot.render(output_path, cleanup=True)

In [ ]:
#graph_model


class Graph:
    def __init__(self):
        self.source_tables = set()
        self.lookup_tables = set()
        self.processing_nodes = set()
        self.output_nodes = set()

        self.edges = []  # (source, target, label)

    def add_source(self, name):
        self.source_tables.add(name)

    def add_lookup(self, name):
        self.lookup_tables.add(name)

    def add_processing(self, name):
        self.processing_nodes.add(name)

    def add_output(self, name):
        self.output_nodes.add(name)

    def add_edge(self, src, tgt, label=None):
        self.edges.append((src, tgt, label))




#transform_summarizer

def summarize_expression(expression: str) -> str:
    if not expression:
        return ""

    expr = expression.lower()

    if "case" in expr:
        return "CASE Logic"
    if "if" in expr and "-99" in expr:
        return "Fallback if -99"
    if "if" in expr and "-97" in expr:
        return "Exception Handling"
    if "*" in expr:
        return "Derived (Multiplication)"
    if "+" in expr:
        return "Derived (Addition)"
    if "nvl" in expr:
        return "Null Handling"
    if "to_date" in expr:
        return "Date Conversion"

    return "Derived"



#graph_builder


from core.graph_model import Graph
from core.transform_summarizer import summarize_expression


def build_graph(lineage_response):
    graph = Graph()

    for stage in lineage_response.stages:

        graph.add_processing(stage.stage_id)

        # INPUTS
        for inp in stage.inputs:
            graph.add_source(inp["dataset_name"])
            graph.add_edge(inp["dataset_name"], stage.stage_id)

        # OUTPUTS
        for out in stage.outputs:
            graph.add_output(out["dataset_name"])
            graph.add_edge(stage.stage_id, out["dataset_name"])

        # COLUMN TRANSFORMATIONS (Skip DIRECT)
        for col in stage.column_lineage:
            if col.transformation_type != "DIRECT":
                label = summarize_expression(col.expression)

                for src in col.source_columns:
                    graph.add_edge(
                        src.split(".")[0],  # table only
                        stage.stage_id,
                        label
                    )

    return graph





#renderer


from graphviz import Digraph


def render_semantic_pipeline(graph, output_path):
    dot = Digraph(format="pdf")
    dot.attr(rankdir="LR")
    dot.attr(nodesep="1.0")
    dot.attr(ranksep="1.2")

    # ---------- SOURCE CLUSTER ----------
    with dot.subgraph(name="cluster_sources") as c:
        c.attr(label="Source Systems", style="rounded")
        for src in graph.source_tables:
            c.node(src, shape="box", style="filled", fillcolor="#1f2c3a")

    # ---------- PROCESSING CLUSTER ----------
    with dot.subgraph(name="cluster_processing") as c:
        c.attr(label="Processing", style="rounded")
        for proc in graph.processing_nodes:
            c.node(proc, shape="box", style="filled", fillcolor="#2d3e50")

    # ---------- OUTPUT CLUSTER ----------
    with dot.subgraph(name="cluster_outputs") as c:
        c.attr(label="Outputs", style="rounded")
        for out in graph.output_nodes:
            c.node(out, shape="box", style="filled", fillcolor="#34495e")

    # ---------- EDGES ----------
    for src, tgt, label in graph.edges:
        if label:
            dot.edge(src, tgt, label=label)
        else:
            dot.edge(src, tgt)

    dot.render(output_path, cleanup=True)



#system prompt


You are a deterministic ETL semantic lineage extractor.

Your goal is to produce a CLEAN transformation model suitable for visualization.

Rules:
1. Do NOT return raw SQL.
2. Do NOT return full expressions.
3. Summarize transformations in short semantic labels (max 6 words).
4. Skip direct column copies.
5. Classify transformations into:
   LOOKUP, DERIVED, CONDITIONAL, FILTER, AGGREGATION.
6. Identify lookup tables separately.
7. Identify exception outputs separately.
8. Return ONLY valid JSON.
9. Do NOT invent tables or joins.
10. If uncertain, omit the transformation.


#user prompt

Analyze the following pseudocode.

Extract a semantic transformation model for diagram generation.

Return JSON in this format:

{
  "stages": [
    {
      "stage_id": "",
      "stage_type": "",
      "source_tables": [],
      "lookup_tables": [],
      "output_tables": [],
      "transformations": [
        {
          "target_column": "",
          "source_tables": [],
          "transformation_category": "",
          "summary_label": ""
        }
      ],
      "output_conditions": [
        {
          "condition": "",
          "label": ""
        }
      ]
    }
  ]
}

Guidelines:
- Skip direct mappings.
- Summarize long logic.
- Keep labels concise.
- Do not include raw SQL.


#error

   main()
    ~~~~^^
  File "/home/admin/shraddha_code_repo/mermaid_llm/main.py", line 19, in main
    lineage = agent.extract_lineage(pseudocode)
  File "/home/admin/shraddha_code_repo/mermaid_llm/agent/llm_agent.py", line 38, in extract_lineage
    return LineageResponse(**parsed_json)
  File "/home/admin/shraddha_code_repo/mermaid_llm/mer_venv/lib/python3.13/site-packages/pydantic/main.py", line 250, in __init__
    validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)
pydantic_core._pydantic_core.ValidationError: 30 validation errors for LineageResponse
stages.0.inputs
  Field required [type=missing, input_value={'stage_id': 'ORA_Ext_Veh...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.0.outputs
  Field required [type=missing, input_value={'stage_id': 'ORA_Ext_Veh...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.0.column_lineage
  Field required [type=missing, input_value={'stage_id': 'ORA_Ext_Veh...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.0.joins
  Field required [type=missing, input_value={'stage_id': 'ORA_Ext_Veh...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.0.filters
  Field required [type=missing, input_value={'stage_id': 'ORA_Ext_Veh...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.1.inputs
  Field required [type=missing, input_value={'stage_id': 'Ora_StgVehi...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.1.outputs
  Field required [type=missing, input_value={'stage_id': 'Ora_StgVehi...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.1.column_lineage
  Field required [type=missing, input_value={'stage_id': 'Ora_StgVehi...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.1.joins
  Field required [type=missing, input_value={'stage_id': 'Ora_StgVehi...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.1.filters
  Field required [type=missing, input_value={'stage_id': 'Ora_StgVehi...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.2.inputs
  Field required [type=missing, input_value={'stage_id': 'HF_FACT_VOR...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.2.outputs
  Field required [type=missing, input_value={'stage_id': 'HF_FACT_VOR...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.2.column_lineage
  Field required [type=missing, input_value={'stage_id': 'HF_FACT_VOR...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.2.joins
  Field required [type=missing, input_value={'stage_id': 'HF_FACT_VOR...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.2.filters
  Field required [type=missing, input_value={'stage_id': 'HF_FACT_VOR...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.3.inputs
  Field required [type=missing, input_value={'stage_id': 'Tfm_LoadRec... for invalid SUPP_SK'}]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.3.outputs
  Field required [type=missing, input_value={'stage_id': 'Tfm_LoadRec... for invalid SUPP_SK'}]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.3.column_lineage
  Field required [type=missing, input_value={'stage_id': 'Tfm_LoadRec... for invalid SUPP_SK'}]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.3.joins
  Field required [type=missing, input_value={'stage_id': 'Tfm_LoadRec... for invalid SUPP_SK'}]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.3.filters
  Field required [type=missing, input_value={'stage_id': 'Tfm_LoadRec... for invalid SUPP_SK'}]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.4.inputs
  Field required [type=missing, input_value={'stage_id': 'HF_SMR_VEHI...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.4.outputs
  Field required [type=missing, input_value={'stage_id': 'HF_SMR_VEHI...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.4.column_lineage
  Field required [type=missing, input_value={'stage_id': 'HF_SMR_VEHI...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.4.joins
  Field required [type=missing, input_value={'stage_id': 'HF_SMR_VEHI...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.4.filters
  Field required [type=missing, input_value={'stage_id': 'HF_SMR_VEHI...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.5.inputs
  Field required [type=missing, input_value={'stage_id': 'Supp_VOR_Ex...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.5.outputs
  Field required [type=missing, input_value={'stage_id': 'Supp_VOR_Ex...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.5.column_lineage
  Field required [type=missing, input_value={'stage_id': 'Supp_VOR_Ex...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.5.joins
  Field required [type=missing, input_value={'stage_id': 'Supp_VOR_Ex...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.5.filters
  Field required [type=missing, input_value={'stage_id': 'Supp_VOR_Ex...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing




In [ ]:
You are a deterministic ETL structure parser.

Your job is NOT to interpret business meaning.
Your job is to extract only explicitly present structural information.

Follow this two-phase strategy strictly:

PHASE 1 — STRUCTURAL EXTRACTION (Deterministic)

Using only explicit patterns in the pseudocode, extract:

1. Stage ID (exact name)
2. Stage Type
3. Input datasets (from "Input: ← dataset_x")
4. Output datasets (from "Output: → dataset_x")
5. Source tables (only from FROM clause)
6. Join tables (only from JOIN / LEFT JOIN / INNER JOIN keywords)
7. Lookup tables (only if stage type is Hash / Lookup OR detected LEFT JOIN)
8. Filters (only from WHERE clause)
9. Stage variables (if explicitly defined)
10. Constraints (if explicitly defined)

Rules:
- Do NOT infer missing tables.
- Do NOT assume relationships.
- Do NOT expand aliases unless explicitly defined.
- Do NOT create tables that do not exist in the text.
- Do NOT summarize yet.
- If something is not explicitly present, return null.

PHASE 2 — LOGIC SUMMARIZATION (Controlled)

After structural extraction:

Summarize only explicitly written logic for:

- CASE expressions
- Fallback logic (e.g., IF X = -99 then Y)
- Conditional assignments
- Derived supplier logic
- Exception routing conditions

Rules:
- No business interpretation.
- No inferred relationships.
- Keep summaries under 15 words each.
- Do not rewrite SQL.
- Do not generate new logic.




You are a deterministic ETL lineage extractor.

Your task is to extract only explicitly present structural and transformation information.

STRICT RULES:

STRUCTURAL EXTRACTION PHASE:
- Identify stage_id exactly as written.
- Identify stage_type exactly as written.
- Identify input datasets only from explicit "Input:" lines.
- Identify output datasets only from explicit "Output:" lines.
- Identify source tables only from FROM clause.
- Identify join tables only from explicit JOIN keywords.
- Identify lookup tables only if:
    - Stage type indicates lookup/hash
    - OR table appears in LEFT JOIN
- Identify filters only from WHERE clause.
- Identify constraints only if explicitly defined.
- Do NOT infer relationships.
- Do NOT expand table aliases unless explicitly defined.
- Do NOT assume business meaning.
- If information is not explicitly present, return null.

TRANSFORMATION SUMMARIZATION PHASE:
- Summarize only explicitly defined transformation logic.
- Summarize CASE expressions.
- Summarize fallback logic (e.g., IF X = -99 then Y).
- Summarize conditional assignments.
- Classify each transformation into:
    LOOKUP, DERIVED, CONDITIONAL, FILTER, AGGREGATION.
- Skip direct column copies.
- Use short labels (max 6 words).
- Do NOT include raw SQL.
- Do NOT include full expressions.
- Do NOT invent transformations.
- If uncertain, omit the transformation.

OUTPUT RULES:
- Return ONLY valid JSON.
- No commentary.
- No explanations.
- No markdown.
- Validate that every table and dataset appears in the input text.
- If structural uncertainty exists, omit that element.

Before returning output:
Verify that every extracted table name, dataset name, and stage name
exists verbatim in the provided text.
If any fabricated element is detected, remove it.

Additionally return:
"graph_nodes": []
"graph_edges": []

Edges must include:
{
  "from": "",
  "to": "",
  "type": "source | lookup | output | exception"
}

In [ ]:
#new_agent_template_for_mermaid

#models.py

class LineageModel:
    def __init__(self):
        self.sources = []
        self.joins = []
        self.filters = []
        self.derived_columns = []
        self.aggregations = []
        self.target = None

    def to_dict(self):
        return {
            "sources": self.sources,
            "joins": self.joins,
            "filters": self.filters,
            "derived_columns": self.derived_columns,
            "aggregations": self.aggregations,
            "target": self.target
        }



  #parser


  import re
from core.models import LineageModel

def parse_pseudocode(text: str) -> LineageModel:
    model = LineageModel()

    lines = text.split("\n")

    for line in lines:
        line = line.strip().lower()

        if "read" in line:
            table = line.replace("read", "").strip()
            model.sources.append(table)

        if "join" in line:
            model.joins.append(line)

        if "filter" in line:
            model.filters.append(line)

        if "create" in line:
            model.derived_columns.append(line)

        if "write" in line:
            model.target = line.replace("write into", "").strip()

    return model



    #utils


    def validate_model(model):
    if not model.sources:
        raise ValueError("No source tables found.")

    if not model.target:
        raise ValueError("No target table defined.")


#graph_engine


import networkx as nx
from graphviz import Digraph

def build_graph(model):
    G = nx.DiGraph()

    for source in model.sources:
        G.add_node(source, type="source")

    G.add_node(model.target, type="target")

    for source in model.sources:
        G.add_edge(source, model.target)

    return G


def render_graph(G, output_path="data/output/lineage_diagram"):
    dot = Digraph()

    for node in G.nodes:
        dot.node(node)

    for edge in G.edges:
        dot.edge(edge[0], edge[1])

    dot.render(output_path, format="png", cleanup=True)



#prompts

def high_level_prompt(structured_json):
    return f"""
Generate a high-level architecture summary from this lineage data:

{structured_json}
"""



#llm_client

from openai import OpenAI
import os

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def generate_response(prompt):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content



#lineage_agent


from core.parser import parse_pseudocode
from core.utils import validate_model
from core.graph_engine import build_graph, render_graph
from llm.prompts import high_level_prompt
from llm.llm_client import generate_response

class LineageAgent:

    def run(self, pseudocode: str):

        # 1️⃣ Parse
        model = parse_pseudocode(pseudocode)

        # 2️⃣ Validate
        validate_model(model)

        structured_json = model.to_dict()

        # 3️⃣ Build Graph
        graph = build_graph(model)
        render_graph(graph)

        # 4️⃣ Generate High-Level Summary
        prompt = high_level_prompt(structured_json)
        high_level_summary = generate_response(prompt)

        return {
            "structured_data": structured_json,
            "high_level_summary": high_level_summary
        }


#main



from agent.lineage_agent import LineageAgent

def main():

    with open("data/input/sample_pseudo.txt", "r") as f:
        pseudocode = f.read()

    agent = LineageAgent()
    result = agent.run(pseudocode)

    print("\n=== Structured JSON ===")
    print(result["structured_data"])

    print("\n=== High Level Summary ===")
    print(result["high_level_summary"])


if __name__ == "__main__":
    main()




# sample_pseudo

# Read customers
# Read orders
# Join customers and orders on customer_id
# Filter order_date > 2024
# Create total_amount = qty * price
# Write into customer_sales


In [ ]:
# Now replace llm/llm_client.py with this:

import os
from dotenv import load_dotenv
from openai import AzureOpenAI

load_dotenv()

class AzureLLMClient:

    def __init__(self):
        self.client = AzureOpenAI(
            api_key=os.getenv("AZURE_OPENAI_API_KEY"),
            azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
            api_version=os.getenv("AZURE_OPENAI_API_VERSION")
        )

        self.deployment = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")

    def generate_response(self, prompt: str) -> str:

        response = self.client.chat.completions.create(
            model=self.deployment,
            messages=[
                {"role": "system", "content": "You are a deterministic ETL lineage summarization engine."},
                {"role": "user", "content": prompt}
            ],
            temperature=0,      # deterministic
            top_p=0.95,
            max_tokens=800
        )

        return response.choices[0].message.content.strip()




# ✅ 4️⃣ Update Agent to Use Azure Client

# Update agent/lineage_agent.py:

from llm.llm_client import AzureLLMClient
from llm.prompts import high_level_prompt

class LineageAgent:

    def __init__(self):
        self.llm = AzureLLMClient()

    def run(self, pseudocode: str):

        model = parse_pseudocode(pseudocode)
        validate_model(model)

        structured_json = model.to_dict()

        graph = build_graph(model)
        render_graph(graph)

        prompt = high_level_prompt(structured_json)
        high_level_summary = self.llm.generate_response(prompt)

        return {
            "structured_data": structured_json,
            "high_level_summary": high_level_summary
        }
# 🔒 5️⃣ Make It More Deterministic (Enterprise Mode)

# If you want stronger guardrails:

# Add strict system instruction:

{"role": "system", "content": """
You are a deterministic ETL lineage summarizer.
Only summarize information present in the provided JSON.
Do NOT assume missing joins, columns, or filters.
If data is missing, state 'Not explicitly defined'.
"""}



In [ ]:
# core/parser.py

import re
from core.models import LineageModel


def parse_pseudocode(text: str) -> LineageModel:
    """
    Deterministic parser for AETL-style pseudocode.
    Extracts:
    - Sources
    - Joins
    - Filters
    - Derived Columns
    - Aggregations
    - Group By
    - Target
    """

    model = LineageModel()

    lines = text.split("\n")

    for raw_line in lines:
        line = raw_line.strip()

        if not line:
            continue

        lower = line.lower()

        # -------------------------
        # READ SOURCE
        # -------------------------
        if lower.startswith("read"):
            table = line.split(" ", 1)[1].strip()
            model.sources.append(table)

        # -------------------------
        # JOIN EXTRACTION
        # Example:
        # Join customers.customer_id = orders.customer_id
        # -------------------------
        elif lower.startswith("join"):
            join_match = re.search(
                r"(\w+)\.(\w+)\s*=\s*(\w+)\.(\w+)",
                line
            )

            if join_match:
                left_table, left_col, right_table, right_col = join_match.groups()

                model.joins.append({
                    "type": "INNER",  # default
                    "left_table": left_table,
                    "left_column": left_col,
                    "right_table": right_table,
                    "right_column": right_col
                })

        # -------------------------
        # FILTER
        # -------------------------
        elif lower.startswith("filter"):
            condition = line.split(" ", 1)[1].strip()
            model.filters.append(condition)

        # -------------------------
        # DERIVED COLUMN
        # Example:
        # Create total_amount = orders.qty * orders.price
        # -------------------------
        elif lower.startswith("create"):
            expr = line.split("create", 1)[1].strip()

            if "=" in expr:
                target_col, formula = expr.split("=", 1)

                target_col = target_col.strip()
                formula = formula.strip()

                model.derived_columns.append({
                    "target_column": target_col,
                    "expression": formula
                })

                # Extract source columns in expression
                source_cols = re.findall(r"\w+\.\w+", formula)
                model.column_lineage[target_col] = source_cols

        # -------------------------
        # AGGREGATION
        # Example:
        # Aggregate SUM(orders.qty) as total_qty
        # -------------------------
        elif lower.startswith("aggregate"):
            agg_match = re.search(
                r"(sum|count|avg|min|max)\((\w+\.\w+)\)\s+as\s+(\w+)",
                line,
                re.IGNORECASE
            )

            if agg_match:
                func, source_col, target_col = agg_match.groups()

                model.aggregations.append({
                    "function": func.upper(),
                    "source_column": source_col,
                    "target_column": target_col
                })

                model.column_lineage[target_col] = [source_col]

        # -------------------------
        # GROUP BY
        # -------------------------
        elif lower.startswith("group by"):
            cols = line.split("group by", 1)[1]
            group_cols = [c.strip() for c in cols.split(",")]
            model.group_by.extend(group_cols)

        # -------------------------
        # TARGET
        # -------------------------
        elif lower.startswith("write into"):
            model.target = line.split("write into", 1)[1].strip()

    return model


#graph_engine

# core/graph_engine.py

import networkx as nx
from graphviz import Digraph


def build_high_level_graph(model):
    """
    Source -> Transformation -> Target
    """

    G = nx.DiGraph()

    transformation_node = "TRANSFORMATION"

    G.add_node(transformation_node)

    for src in model.sources:
        G.add_node(src)
        G.add_edge(src, transformation_node)

    if model.target:
        G.add_node(model.target)
        G.add_edge(transformation_node, model.target)

    return G

def build_detailed_graph(model):
    """
    Detailed lineage graph:
    - Tables
    - Join relationships
    - Derived columns
    - Aggregations
    - Column lineage
    """

    G = nx.DiGraph()

    # -------------------------
    # Add source & target tables
    # -------------------------
    for src in model.sources:
        G.add_node(src, type="table")

    if model.target:
        G.add_node(model.target, type="table")

    # -------------------------
    # Add join relationships
    # -------------------------
    for join in model.joins:
        left = f"{join['left_table']}.{join['left_column']}"
        right = f"{join['right_table']}.{join['right_column']}"

        G.add_node(left, type="column")
        G.add_node(right, type="column")

        G.add_edge(left, right, label="JOIN")

    # -------------------------
    # Add derived column lineage
    # -------------------------
    for target_col, source_cols in model.column_lineage.items():

        target_node = f"{model.target}.{target_col}"
        G.add_node(target_node, type="column")

        for src_col in source_cols:
            G.add_node(src_col, type="column")
            G.add_edge(src_col, target_node, label="DERIVED")

    # -------------------------
    # Add aggregation nodes
    # -------------------------
    for agg in model.aggregations:
        agg_node = f"{agg['function']}({agg['source_column']})"

        source_col = agg["source_column"]
        target_col = f"{model.target}.{agg['target_column']}"

        G.add_node(agg_node, type="aggregation")
        G.add_node(target_col, type="column")

        G.add_edge(source_col, agg_node, label="AGG_INPUT")
        G.add_edge(agg_node, target_col, label="AGG_OUTPUT")

    return G


def render_graph(G, output_path):
    """
    Render NetworkX graph using Graphviz
    """

    dot = Digraph()

    for node in G.nodes:
        dot.node(node)

    for edge in G.edges(data=True):
        label = edge[2].get("label", "")
        dot.edge(edge[0], edge[1], label=label)

    dot.render(output_path, format="png", cleanup=True)



#pdf_generator

from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors
from reportlab.lib.units import inch


def generate_pdf(output_path,
                 high_level_img,
                 detailed_img):

    doc = SimpleDocTemplate(output_path)
    styles = getSampleStyleSheet()
    elements = []

    elements.append(Paragraph("AETL Lineage Report", styles["Heading1"]))
    elements.append(Spacer(1, 0.3 * inch))

    elements.append(Paragraph("High-Level Flow Diagram", styles["Heading2"]))
    elements.append(Spacer(1, 0.2 * inch))
    elements.append(Image(high_level_img, width=6*inch, height=3*inch))

    elements.append(Spacer(1, 0.5 * inch))

    elements.append(Paragraph("Detailed Column-Level Diagram", styles["Heading2"]))
    elements.append(Spacer(1, 0.2 * inch))
    elements.append(Image(detailed_img, width=6*inch, height=4*inch))

    doc.build(elements)



#lineage_agent

from core.graph_engine import (
    build_high_level_graph,
    build_detailed_graph,
    render_graph
)
from core.pdf_generator import generate_pdf


def run(self, pseudocode):

    model = parse_pseudocode(pseudocode)
    validate_model(model)

    # High-level
    high_graph = build_high_level_graph(model)
    render_graph(high_graph, "data/output/high_level")

    # Detailed
    detailed_graph = build_detailed_graph(model)
    render_graph(detailed_graph, "data/output/detailed")

    # Generate PDF
    generate_pdf(
        "data/output/lineage_report.pdf",
        "data/output/high_level.png",
        "data/output/detailed.png"
    )



#moidels

class LineageModel:
    def __init__(self):
        self.sources = []
        self.joins = []              # structured join objects
        self.filters = []
        self.derived_columns = []    # non-aggregate computed columns
        self.aggregations = []       # aggregation definitions
        self.group_by = []           # group by columns
        self.column_lineage = {}     # target_column -> source_columns
        self.target = None

    def to_dict(self):
        return {
            "sources": self.sources,
            "joins": self.joins,
            "filters": self.filters,
            "derived_columns": self.derived_columns,
            "aggregations": self.aggregations,
            "group_by": self.group_by,
            "column_lineage": self.column_lineage,
            "target": self.target
        }








